# 07 · Soil properties and conservation practices against the drought penalty (pre-registered)

**Question.** Do surface organic matter (primary), other soil properties (exploratory) and 2017 conservation-practice shares (exploratory) go with smaller drought penalties in Iowa, using the notebook 05 framework unchanged?

**Status.** This cell is the full plan, written and committed **before any result exists**. Every rule below is fixed in advance and is not changed after seeing results. Whatever the outcome, it is reported. Iowa only; corn and soybeans separately; `fips` kept as 5-character text.

Plan committed and pushed as dd6722b before any result was computed.

## Framework (imported, not copied; `soil_tool.py` is not edited)
- Drought penalty and placebo: `soil_tool.compute_penalties(load_iowa_with_lon(), split=True)`, exactly as in notebook 05 Step 2 (normals never cross 2015/2016; drought rows are D2+, `max_drought_severity >= 3`; non-D2+ rows carry the placebo value). Parameters asserted from the module at run time: `WINDOW = 3`, `MIN_NORMAL_YEARS = 2`, `SEV_D2 = 3`, `MIN_COUNTIES_PER_YEAR = 15` (years with at least 15 county-years of the relevant kind with a value), `N_BOOT = 5000`, `BOOT_SEED = 20261017`, 95% range from the 2.5th and 97.5th percentiles of year-bootstrap means. Nothing is re-derived.
- Per-year regression and bootstrap: `soil_tool.penalty_storage_results` (per year, OLS slope of the penalty on the variable plus controls across that year's counties, averaged over years, year-bootstrap range; separately for drought rows and placebo rows). It is hard-wired to regress on the storage column `aws_100cm_mm` in units of 10 mm (`AWS_UNIT`) with the specifications in `PEN_SPECS`. **Thin wrapper, declared now:** the notebook function `run_variable(ia_split, variable, specs, unit)` copies the variable of interest into the `aws_100cm_mm` column of a *copy* of the data (the real storage values are kept under another name and are never used as a control), sets the module attributes `AWS_UNIT = unit` and `PEN_SPECS = specs` for the duration of the call, calls `penalty_storage_results`, and restores the attributes. Year selection, bootstrap, placebo and output format are therefore the module's own.
- Categorical variable (dominant drainage class): `penalty_storage_results` regresses on one variable, so the wrapper `run_categorical` fits, per year, one OLS of the penalty on the class dummies plus the location controls, with the **same** year rule (`MIN_COUNTIES_PER_YEAR`), the same drought/placebo split and the module's own `_year_bootstrap` for each dummy coefficient. Reference class: **Well drained**.
- County variables: surface organic matter, clay, sand, slope and dominant drainage class from `soil_planner/extra_data/iowa_soil_extra.csv`; 2017 practice shares from `soil_planner/extra_data/iowa_conservation_practices.csv`; `aws_100cm_mm` from the master; county centroid **latitude and longitude** computed from `load_counties()` as the mean of the polygon vertices (the same construction as `soil_tool.county_centroid_lon`, extended to latitude).

## Part 1: PRIMARY TEST, declared in advance: surface organic matter
1. Descriptives first: Pearson correlations of organic matter with longitude, latitude, slope and `aws_100cm_mm` across the 99 counties, and mean organic matter by dominant drainage class.
2. Within each drought year with at least 15 D2+ counties with a penalty, regress the drought penalty on surface organic matter, **per +1 percentage point** of organic matter:
   - **A.** organic matter only
   - **B.** + longitude + latitude
   - **C.** + longitude + latitude + slope
3. Year-bootstrap 95% ranges; the same three specifications on the placebo (non-D2+ county-years, years with at least 15 such rows).
4. **Decision rule (fixed now), per crop:** call it **"a clear association"** only if specification **C**'s drought-year range **excludes zero AND** its placebo range **includes zero**. Any other pattern is **"no clear effect"**. Specifications A and B are reported but do not decide.

## Part 2: EXPLORATORY: other soil properties
- Specification **B** (longitude + latitude) for **clay** (per +1 pp), **sand** (per +1 pp), **slope** (per +1 pp) and **dominant drainage class** (dummies for each class present, Well drained as reference), drought versus placebo, in one table labelled **EXPLORATORY**.
- **Test count.** Tests = drought-year estimates in Parts 1 and 2 (placebo estimates are checks, not tests, and are not counted): Part 1 has 3 specifications x 2 crops = 6; Part 2 has 3 continuous variables plus one dummy per non-reference drainage class present, x 2 crops. The notebook reports the total N and the number expected to exclude zero by chance alone at a 95% range, 0.05 x N. No single estimate from Part 2 is treated as a finding.

## Part 3: EXPLORATORY: conservation practices (2017 Census)
- Exposure: **2017** shares only (`no_till_share`, `cover_crop_share`), measured before the outcomes. Outcome: drought penalties in **drought years 2018–2025** that meet the 15-county rule (at least 15 D2+ counties with a penalty in that year). **If fewer than 3 such years qualify for a crop, report that and stop Part 3 for that crop.**
- Within each qualifying year, regress the penalty on the 2017 **no-till share (per +10 percentage points)** and, separately, the 2017 **cover-crop share (per +1 percentage point)**, each with longitude + latitude + slope. Same bootstrap; placebo = non-D2+ county-years in 2018–2025, years with at least 15 such rows.
- **Guardrail (fixed):** control only for variables practices cannot change (location, slope). **Never** control for organic matter, `aws_100cm_mm` or any other soil property that practices may affect; those lie on the pathway from practice to drought loss (Bueno de Mesquita & Fowler, ch. 10; Gelman, Hill & Vehtari, §19.6).
- Interpretation written in advance: counties may have adopted practices **because** of earlier droughts (for example 2012), so even the direction of any association is ambiguous (Bueno de Mesquita & Fowler, ch. 9); farmers choose these practices, so no association here shows a practice effect.

## Summary rules (written after the results, judged by the rules above)
- Part 1: apply the decision rule and state which outcome it gives for each crop.
- If any range excludes zero, say that with this few years the size is likely overstated and should not be quoted as the effect (Gelman, Hill & Vehtari, §4.4, §16.1).
- Parts 2 and 3: patterns only; nothing is claimed from a single estimate.
- Compare with the storage result from notebook 05 (`penalty_storage_results` on `aws_100cm_mm`, recomputed here with the unchanged module settings as the reference row).

## Guardrails
- No specification, unit, threshold, year rule or control set is changed after seeing results; everything above is reported in full.
- `soil_planner/soil_tool.py`, the app, the priority ranking and the evidence tables are not changed. The extra soil and practice data remain display-only everywhere else.
- Nothing under `data/` or `src/` is modified. The notebook runs from the repo root with `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 notebooks/07_soil_and_practices.ipynb`.


## Setup: import the framework, assert the parameters, build the county variables

In [1]:
import sys
sys.path.insert(0, "..")
import contextlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from soil_planner import soil_tool as st      # the framework; not copied, not modified
from src.data_loader import load_counties

pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)

# ---- parameters, pre-registered; asserted against the module ----
assert st.WINDOW == 3 and st.MIN_NORMAL_YEARS == 2 and st.SEV_D2 == 3 and st.MIN_COUNTIES_PER_YEAR == 15
assert st.N_BOOT == 5000 and st.BOOT_SEED == 20261017 and tuple(st.CI) == (2.5, 97.5)
FIPS, YEAR, CROP, PEN, KIND, AWS, LON = st.COL_FIPS, st.COL_YEAR, st.COL_CROP, st.COL_PEN, st.COL_KIND, st.COL_AWS, st.COL_LON
LAT, OM, CLAY, SAND, SLOPE, DRAIN = "lat", "om_surface_pct", "clay_surface_pct", "sand_surface_pct", "slope_pct", "drainage_dominant"
CROPS = dict(st.CROPS)
print(f"WINDOW={st.WINDOW}, MIN_NORMAL_YEARS={st.MIN_NORMAL_YEARS}, D2 code={st.SEV_D2}, min counties per year={st.MIN_COUNTIES_PER_YEAR}, "
      f"N_BOOT={st.N_BOOT}, seed={st.BOOT_SEED}, CI={st.CI}; module AWS_UNIT={st.AWS_UNIT}, PEN_SPECS={list(st.PEN_SPECS)}")

# ---- penalties and placebo, exactly as notebook 05 Step 2 ----
ia = st.compute_penalties(st.load_iowa_with_lon(), split=True)
assert ia[FIPS].map(type).eq(str).all() and ia[FIPS].str.len().eq(5).all()

# ---- county centroid latitude and longitude from load_counties(): mean of polygon vertices ----
def _centroid(feature):
    g = feature["geometry"]
    polys = g["coordinates"] if g["type"] == "Polygon" else [p for mp in g["coordinates"] for p in mp]
    pts = np.array([pt for ring in polys for pt in ring])
    return pts[:, 0].mean(), pts[:, 1].mean()
cent = pd.DataFrame([(str(f["id"]), *_centroid(f)) for f in load_counties()["features"] if str(f["id"]).startswith(st.IOWA_FIPS_PREFIX)],
                    columns=[FIPS, "lon_check", LAT])
assert np.allclose(ia.merge(cent, on=FIPS)[LON], ia.merge(cent, on=FIPS)["lon_check"])   # same construction as soil_tool.county_centroid_lon

# ---- county variables (display-only elsewhere; here they are the test variables) ----
soil = st.load_soil_extra()[[FIPS, OM, CLAY, SAND, SLOPE, DRAIN]]
prac = st.load_conservation_practices()
prac17 = prac[prac[YEAR] == 2017][[FIPS, "no_till_share", "cover_crop_share"]].rename(columns={"no_till_share": "no_till_2017", "cover_crop_share": "cover_crop_2017"})
ia = ia.merge(cent[[FIPS, LAT]], on=FIPS, how="left").merge(soil, on=FIPS, how="left").merge(prac17, on=FIPS, how="left")
for c in [LAT, OM, CLAY, SAND, SLOPE, DRAIN, "no_till_2017", "cover_crop_2017"]:
    assert ia[c].notna().all(), c
county = ia.drop_duplicates(FIPS)[[FIPS, st.COL_COUNTY, LON, LAT, AWS, OM, CLAY, SAND, SLOPE, DRAIN, "no_till_2017", "cover_crop_2017"]].reset_index(drop=True)
print(f"{len(ia):,} Iowa county-years, {len(county)} counties; penalty values: {int(ia[PEN].notna().sum()):,}")

# ---- thin wrappers (declared in the plan) ----
@contextlib.contextmanager
def module_settings(**overrides):
    """Temporarily set soil_tool module attributes at run time, then restore them. The file is not modified."""
    saved = {k: getattr(st, k) for k in overrides}
    try:
        for k, v in overrides.items():
            setattr(st, k, v)
        yield
    finally:
        for k, v in saved.items():
            setattr(st, k, v)

def run_variable(data, variable, specs, unit, label):
    """penalty_storage_results on `variable` instead of storage: the variable is copied into the storage column of a
    copy of the data; the real storage is kept as 'aws_true' and is never a control. unit = slope multiplier
    (1.0 -> per +1 unit of the variable; 0.10 -> per +0.10 = +10 pp of a share)."""
    d = data.copy()
    d["aws_true"] = d[AWS]
    d[AWS] = d[variable]
    with module_settings(AWS_UNIT=unit, PEN_SPECS=specs):
        table, slopes = st.penalty_storage_results(d)
    assert st.AWS_UNIT == 10.0 and list(st.PEN_SPECS) == ["storage only", "storage + longitude"]   # restored
    table.insert(0, "variable", label)
    return table, slopes

def run_categorical(data, cat_col, reference, controls, label):
    """Per year (same year rule), OLS of the penalty on class dummies + controls; the module's own year bootstrap
    for each dummy coefficient; drought and placebo rows separately."""
    classes = [c for c in sorted(data[cat_col].unique()) if c != reference]
    rows = []
    for crop in CROPS.values():
        for kind in ["drought", "placebo"]:
            sub = data[(data[CROP] == crop) & (data[KIND] == kind) & data[PEN].notna()]
            n = sub.groupby(YEAR)[FIPS].nunique()
            sub = sub[sub[YEAR].isin(n[n >= st.MIN_COUNTIES_PER_YEAR].index)]
            coefs = {c: [] for c in classes}
            for y, g in sub.groupby(YEAR):
                X = np.column_stack([np.ones(len(g))] + [(g[cat_col] == c).to_numpy(float) for c in classes] + [g[c].to_numpy(float) for c in controls])
                if np.linalg.matrix_rank(X) < X.shape[1]:
                    continue
                beta, *_ = np.linalg.lstsq(X, g[PEN].to_numpy(float), rcond=None)
                for i, c in enumerate(classes):
                    coefs[c].append(beta[1 + i])
            for c in classes:
                m, lo, hi, k = st._year_bootstrap(coefs[c])
                rows.append({"variable": f"{label}: {c} vs {reference}", "crop": crop, "spec": "B: + longitude + latitude", "kind": kind,
                             "estimate [95% range]": st._fmt(m, lo, hi), "n_years": k, "county_years": len(sub),
                             "years +/-": f"{int((np.array(coefs[c]) > 0).sum())} / {int((np.array(coefs[c]) < 0).sum())}",
                             "excludes_zero": bool(lo > 0 or hi < 0), "_mean": m, "_lo": lo, "_hi": hi})
    return pd.DataFrame(rows)

def wide(table):
    """drought and placebo side by side."""
    t = table.pivot(index=["variable", "crop", "spec"], columns="kind", values=["estimate [95% range]", "n_years", "county_years", "excludes_zero"])
    t = t.swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)[["drought", "placebo"]]
    return t.loc[table.drop_duplicates(["variable", "crop", "spec"]).set_index(["variable", "crop", "spec"]).index]

WINDOW=3, MIN_NORMAL_YEARS=2, D2 code=3, min counties per year=15, N_BOOT=5000, seed=20261017, CI=(2.5, 97.5); module AWS_UNIT=10.0, PEN_SPECS=['storage only', 'storage + longitude']


4,907 Iowa county-years, 99 counties; penalty values: 4,360


## Part 1: PRIMARY TEST: surface organic matter

### 1a. Descriptives: what organic matter travels with

In [2]:
corr = county[[OM, LON, LAT, SLOPE, AWS]].corr().loc[[OM], [LON, LAT, SLOPE, AWS]].round(2)
corr.index = ["Pearson r with organic matter"]
display(corr)
display(county.groupby(DRAIN)[OM].agg(counties="size", mean_organic_matter="mean", sd="std").round(2))
print(f"Organic matter across the 99 counties: mean {county[OM].mean():.2f}%, sd {county[OM].std():.2f}, range {county[OM].min():.2f}-{county[OM].max():.2f}")

,lon,lat,slope_pct,aws_100cm_mm
Pearson r with organic matter,-0.24,0.66,-0.81,0.03


,counties,mean_organic_matter,sd
drainage_dominant,,,
Moderately well drained,16,3.28,0.51
Poorly drained,20,6.07,1.10
Somewhat poorly drained,20,3.86,1.15
Well drained,43,3.53,0.72


Organic matter across the 99 counties: mean 4.07%, sd 1.34, range 2.31-8.75


### 1b. Drought penalty on organic matter, per +1 percentage point, drought years vs placebo

Years with at least 15 county-years of the relevant kind; mean per-year slope with a year-bootstrap 95% range. Positive = more organic matter, smaller penalty.

In [3]:
SPECS_P1 = {"A: organic matter only": [], "B: + longitude + latitude": [LON, LAT], "C: + longitude + latitude + slope": [LON, LAT, SLOPE]}
p1, p1_slopes = run_variable(ia, OM, SPECS_P1, 1.0, "organic matter (per +1 pp)")
display(wide(p1))
print("Per-year slopes, specification C:")
for crop in CROPS.values():
    t = p1_slopes[(crop, "drought", "C: + longitude + latitude + slope")].rename(columns=lambda c: f"drought {c}") \
        .join(p1_slopes[(crop, "placebo", "C: + longitude + latitude + slope")].rename(columns=lambda c: f"placebo {c}"), how="outer")
    print(crop); display(t.round(2).T)

kind                                                                                 drought                                                  placebo                                   
                                                                        estimate [95% range] n_years county_years excludes_zero  estimate [95% range] n_years county_years excludes_zero
variable                   crop     spec                                                                                                                                                
organic matter (per +1 pp) CORN     A: organic matter only             +3.55 [-0.75, +10.16]      13          681         False  +0.31 [-0.92, +1.82]      23         1497         False
                                    B: + longitude + latitude           +2.23 [-0.29, +5.25]      13          681         False  +0.16 [-0.69, +1.13]      23         1497         False
                                    C: + longitude + latitude + slope   +2.54 [-0.51, +5.65]      13          681         False  -0.05 [-1.56, +1.45]      23         1497         False
                           SOYBEANS A: organic matter only              +0.41 [-0.97, +1.66]      14          720         False  +0.81 [-0.30, +2.06]      21         1362         False
                                    B: + longitude + latitude           -0.45 [-2.18, +1.15]      14          720         False  +0.43 [-0.34, +1.21]      21         1362         False
                                    C: + longitude + latitude + slope   -0.45 [-3.70, +2.07]      14          720         False  -0.18 [-1.61, +1.10]      21         1362         False

Per-year slopes, specification C:
CORN


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
drought n_counties,70.00,NaN,NaN,92.00,NaN,26.00,51.00,NaN,NaN,NaN,NaN,16.00,99.00,64.00,NaN,NaN,NaN,26.00,NaN,NaN,48.00,57.00,41.00,73.0,18.00,NaN
drought slope_per_10mm,4.09,NaN,NaN,-0.08,NaN,12.54,3.07,NaN,NaN,NaN,NaN,14.29,0.15,-8.69,NaN,NaN,NaN,3.09,NaN,NaN,3.56,1.11,-1.97,3.7,-1.85,NaN
placebo n_counties,16.00,86.00,85.00,NaN,99.00,73.00,48.00,89.0,99.00,99.00,99.00,83.00,NaN,35.00,74.00,34.00,81.00,69.00,78.00,88.00,41.00,25.00,47.00,NaN,30.00,19.00
placebo slope_per_10mm,-9.59,-1.82,0.59,NaN,0.18,0.92,-0.21,4.3,-3.64,0.94,-1.44,1.48,NaN,-2.96,1.49,1.56,1.95,-1.12,-4.88,1.07,2.59,9.21,1.20,NaN,1.95,-4.98


SOYBEANS


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
drought n_counties,70.00,NaN,NaN,92.00,NaN,47.00,51.00,NaN,NaN,NaN,NaN,33.00,99.00,65.00,NaN,NaN,NaN,30.00,17.00,NaN,42.00,50.00,50.00,52.0,22.00
drought slope_per_10mm,3.20,NaN,NaN,1.78,NaN,3.75,2.61,NaN,NaN,NaN,NaN,-3.60,0.35,-9.08,NaN,NaN,NaN,1.34,-15.81,NaN,0.99,1.60,-0.54,5.8,1.34
placebo n_counties,16.00,86.0,85.0,NaN,99.00,52.00,48.00,89.0,99.00,99.00,99.00,66.00,NaN,34.00,70.00,33.00,79.00,65.00,77.00,80.00,37.00,18.00,31.00,NaN,NaN
placebo slope_per_10mm,-9.11,-1.2,1.2,NaN,-0.74,0.33,0.13,3.7,-2.76,0.15,-0.93,-0.43,NaN,-3.59,-0.91,3.36,2.54,1.28,-4.94,0.31,2.13,6.15,-0.51,NaN,NaN


### 1c. Pre-registered decision rule, applied

"A clear association" only if specification C's drought range excludes zero **and** its placebo range includes zero; otherwise "no clear effect".

In [4]:
def decide(table, spec):
    out = {}
    for crop in CROPS.values():
        d = table[(table["crop"] == crop) & (table["spec"] == spec) & (table["kind"] == "drought")].iloc[0]
        p = table[(table["crop"] == crop) & (table["spec"] == spec) & (table["kind"] == "placebo")].iloc[0]
        clear = bool(d["excludes_zero"] and not p["excludes_zero"])
        out[crop] = {"drought estimate": d["estimate [95% range]"], "drought n_years": int(d["n_years"]), "drought range excludes zero": bool(d["excludes_zero"]),
                     "placebo estimate": p["estimate [95% range]"], "placebo range includes zero": bool(not p["excludes_zero"]),
                     "DECISION": "a clear association" if clear else "no clear effect"}
    return pd.DataFrame(out).T
P1_DECISION = decide(p1, "C: + longitude + latitude + slope")
display(P1_DECISION)
for crop, r in P1_DECISION.iterrows():
    print(f"{crop}: {r['DECISION']}")

,drought estimate,drought n_years,drought range excludes zero,placebo estimate,placebo range includes zero,DECISION
CORN,"+2.54 [-0.51, +5.65]",13,False,"-0.05 [-1.56, +1.45]",True,no clear effect
SOYBEANS,"-0.45 [-3.70, +2.07]",14,False,"-0.18 [-1.61, +1.10]",True,no clear effect


CORN: no clear effect
SOYBEANS: no clear effect


## Part 2: EXPLORATORY: other soil properties

Specification B (longitude + latitude) for clay, sand and slope (per +1 pp) and the dominant drainage class (dummies, Well drained as reference). **EXPLORATORY**: patterns only; no single estimate is a finding.

In [5]:
SPEC_B = {"B: + longitude + latitude": [LON, LAT]}
p2_parts = [run_variable(ia, CLAY, SPEC_B, 1.0, "clay (per +1 pp)")[0],
            run_variable(ia, SAND, SPEC_B, 1.0, "sand (per +1 pp)")[0],
            run_variable(ia, SLOPE, SPEC_B, 1.0, "slope (per +1 pp)")[0],
            run_categorical(ia, DRAIN, "Well drained", [LON, LAT], "drainage class")]
p2 = pd.concat(p2_parts, ignore_index=True)
display(Markdown("**EXPLORATORY** — drought vs placebo, specification B"))
display(wide(p2))
n_tests_p1 = int((p1["kind"] == "drought").sum())
n_tests_p2 = int((p2["kind"] == "drought").sum())
N_TESTS = n_tests_p1 + n_tests_p2
n_excl = int(((p1["kind"] == "drought") & p1["excludes_zero"]).sum() + ((p2["kind"] == "drought") & p2["excludes_zero"]).sum())
n_excl_plc = int(((p1["kind"] == "placebo") & p1["excludes_zero"]).sum() + ((p2["kind"] == "placebo") & p2["excludes_zero"]).sum())
print(f"Tests (drought-year estimates) in Parts 1 and 2: {n_tests_p1} + {n_tests_p2} = {N_TESTS}. "
      f"Expected to exclude zero by chance alone at a 95% range: {0.05 * N_TESTS:.1f}. Observed: {n_excl} (placebo side: {n_excl_plc} of {N_TESTS}).")

**EXPLORATORY** — drought vs placebo, specification B

kind                                                                                                     drought                                                  placebo                                   
                                                                                            estimate [95% range] n_years county_years excludes_zero  estimate [95% range] n_years county_years excludes_zero
variable                                                crop     spec                                                                                                                                       
clay (per +1 pp)                                        CORN     B: + longitude + latitude  +0.46 [-0.09, +0.99]      13          681         False  +0.11 [-0.10, +0.31]      23         1497         False
                                                        SOYBEANS B: + longitude + latitude  +0.45 [+0.01, +0.90]      14          720          True  +0.15 [-0.22, +0.46]      21         1362         False
sand (per +1 pp)                                        CORN     B: + longitude + latitude  +0.07 [-0.08, +0.22]      13          681         False  +0.01 [-0.08, +0.10]      23         1497         False
                                                        SOYBEANS B: + longitude + latitude  -0.06 [-0.27, +0.13]      14          720         False  +0.04 [-0.08, +0.17]      21         1362         False
slope (per +1 pp)                                       CORN     B: + longitude + latitude  -0.63 [-1.26, +0.06]      13          681         False  -0.02 [-0.19, +0.15]      23         1497         False
                                                        SOYBEANS B: + longitude + latitude  -0.12 [-0.52, +0.34]      14          720         False  -0.12 [-0.36, +0.12]      21         1362         False
drainage class: Moderately well drained vs Well drained CORN     B: + longitude + latitude  +2.97 [-0.09, +6.57]      13          681         False  +0.24 [-1.83, +2.37]      21         1497         False
drainage class: Poorly drained vs Well drained          CORN     B: + longitude + latitude  +3.77 [+0.28, +6.92]      13          681          True  +1.22 [-1.18, +3.67]      21         1497         False
drainage class: Somewhat poorly drained vs Well drained CORN     B: + longitude + latitude  +2.10 [-2.51, +6.71]      13          681         False  +0.16 [-2.12, +2.41]      21         1497         False
drainage class: Moderately well drained vs Well drained SOYBEANS B: + longitude + latitude  -1.50 [-3.98, +0.97]      13          720         False  -0.15 [-2.11, +1.73]      19         1362         False
drainage class: Poorly drained vs Well drained          SOYBEANS B: + longitude + latitude  +3.06 [-0.14, +6.08]      13          720         False  +0.21 [-3.06, +3.15]      19         1362         False
drainage class: Somewhat poorly drained vs Well drained SOYBEANS B: + longitude + latitude  -1.78 [-7.58, +4.43]      13          720         False  -0.28 [-2.87, +2.18]      19         1362         False

Tests (drought-year estimates) in Parts 1 and 2: 6 + 12 = 18. Expected to exclude zero by chance alone at a 95% range: 0.9. Observed: 2 (placebo side: 0 of 18).


## Part 3: EXPLORATORY: conservation practices (2017 Census shares)

Exposure measured in 2017, outcomes in drought years 2018–2025 that meet the 15-county rule; stop for a crop with fewer than 3 such years. Controls: longitude, latitude, slope only. **Never** organic matter, storage or other soil properties that practices may affect (on the pathway: Bueno de Mesquita & Fowler, ch. 10; Gelman, Hill & Vehtari, §19.6).

In [6]:
P3_YEARS = range(2018, 2026)
ia3 = ia[ia[YEAR].isin(P3_YEARS)]
gate3 = {}
for crop_label, crop in CROPS.items():
    d = ia3[(ia3[CROP] == crop) & (ia3[KIND] == "drought") & ia3[PEN].notna()].groupby(YEAR)[FIPS].nunique()
    ok = d[d >= st.MIN_COUNTIES_PER_YEAR]
    gate3[crop] = {"qualifying drought years": {int(y): int(n) for y, n in ok.items()}, "n": len(ok), "proceed": len(ok) >= 3}
    print(f"{crop_label}: drought years 2018-2025 with >= {st.MIN_COUNTIES_PER_YEAR} D2+ counties with a penalty: {gate3[crop]['qualifying drought years']} -> "
          + ("proceed" if gate3[crop]["proceed"] else "STOP Part 3 for this crop (fewer than 3 years)"))
P3_CROPS = {k: v for k, v in CROPS.items() if gate3[v]["proceed"]}
p3 = None
if P3_CROPS:
    SPEC_P3 = {"longitude + latitude + slope": [LON, LAT, SLOPE]}
    with module_settings(CROPS=P3_CROPS):
        ia3p = ia3[ia3[CROP].isin(P3_CROPS.values())]
        p3 = pd.concat([run_variable(ia3p, "no_till_2017", SPEC_P3, 0.10, "2017 no-till share (per +10 pp)")[0],
                        run_variable(ia3p, "cover_crop_2017", SPEC_P3, 0.01, "2017 cover-crop share (per +1 pp)")[0]], ignore_index=True)
    assert st.CROPS == CROPS
    display(Markdown("**EXPLORATORY** — outcomes 2018–2025, drought vs placebo"))
    display(wide(p3))
    print("2017 shares across the 99 counties: no-till mean {:.1%} (sd {:.1%}); cover crop mean {:.1%} (sd {:.1%}); corr(no-till, cover crop) = {:.2f}".format(
        county["no_till_2017"].mean(), county["no_till_2017"].std(), county["cover_crop_2017"].mean(), county["cover_crop_2017"].std(),
        county["no_till_2017"].corr(county["cover_crop_2017"])))
else:
    print("Part 3 not run: no crop has 3 qualifying years.")

Corn: drought years 2018-2025 with >= 15 D2+ counties with a penalty: {2020: 48, 2021: 57, 2022: 41, 2023: 73, 2024: 18} -> proceed
Soybeans: drought years 2018-2025 with >= 15 D2+ counties with a penalty: {2018: 17, 2020: 42, 2021: 50, 2022: 50, 2023: 52, 2024: 22} -> proceed


**EXPLORATORY** — outcomes 2018–2025, drought vs placebo

kind                                                                                  drought                                                  placebo                                   
                                                                         estimate [95% range] n_years county_years excludes_zero  estimate [95% range] n_years county_years excludes_zero
variable                          crop     spec                                                                                                                                          
2017 no-till share (per +10 pp)   CORN     longitude + latitude + slope  +0.21 [-1.26, +1.60]       5          237         False  -0.33 [-1.47, +0.79]       7          328         False
                                  SOYBEANS longitude + latitude + slope  -0.69 [-1.62, +0.26]       6          233         False  +0.22 [-0.22, +0.71]       5          243         False
2017 cover-crop share (per +1 pp) CORN     longitude + latitude + slope  +0.18 [-0.06, +0.47]       5          237         False  +0.02 [-0.45, +0.51]       7          328         False
                                  SOYBEANS longitude + latitude + slope  -0.14 [-0.32, +0.12]       6          233         False  +0.10 [-0.30, +0.57]       5          243         False

2017 shares across the 99 counties: no-till mean 38.3% (sd 23.6%); cover crop mean 4.2% (sd 2.5%); corr(no-till, cover crop) = 0.27


### Reference: storage result from notebook 05 (unchanged module settings, per +10 mm)

In [7]:
ref, _ = st.penalty_storage_results(ia)
ref.insert(0, "variable", "soil water storage (per +10 mm), notebook 05")
display(wide(ref))

kind                                                                                    drought                                                  placebo                                   
                                                                           estimate [95% range] n_years county_years excludes_zero  estimate [95% range] n_years county_years excludes_zero
variable                                     crop     spec                                                                                                                                 
soil water storage (per +10 mm), notebook 05 CORN     storage only         +0.83 [-1.57, +3.44]      13          681         False  -0.30 [-1.13, +0.51]      23         1497         False
                                                      storage + longitude  +0.88 [-1.72, +3.75]      13          681         False  -0.24 [-1.09, +0.64]      23         1497         False
                                             SOYBEANS storage only         +1.81 [-0.08, +3.73]      14          720         False  -0.30 [-1.26, +0.60]      21         1362         False
                                                      storage + longitude  +1.38 [-0.92, +3.38]      14          720         False  +0.13 [-0.93, +1.23]      21         1362         False

## Summary (plain language, judged by the pre-registered rules)

**Part 1, primary test: "no clear effect" for both crops.** Under the fixed decision rule (specification C, with longitude, latitude and slope), the drought-year range for surface organic matter includes zero for corn (+2.5 pp of penalty per +1 pp of organic matter, range −0.5 to +5.7, 13 drought years) and for soybeans (−0.5, range −3.7 to +2.1, 14 years). Both placebo ranges include zero, as they should. Corn leans the "right" way (more organic matter, smaller penalty) and soybeans do not lean at all; neither is separated from zero. The descriptives explain why controls matter here: organic matter runs with the terrain (correlation −0.81 with slope, +0.66 with latitude) and is highest in poorly drained counties (mean 6.1% against 3.5% in well-drained ones), while it is unrelated to soil water storage (+0.03). The raw corn estimate (specification A, +3.6) shrinks by a third once location is added, which is the pattern of a variable standing in for region.

**Part 2, exploratory: two of 18 ranges exclude zero, against 0.9 expected by chance; the placebo side shows 0 of 18.** The two are soybean clay (+0.45 per +1 pp, range +0.01 to +0.90) and corn "poorly drained vs well drained" (+3.8, range +0.3 to +6.9); both clear zero narrowly. Patterns worth noting, not findings: clay leans positive in both crops; steeper counties lean toward larger corn penalties (−0.6 per +1 pp slope, range includes zero); poorly drained counties lean toward smaller penalties in both crops. With 18 tests, about one range would exclude zero by chance alone, so two is not evidence of anything on its own, and no single estimate here should be quoted. (Two placebo years were dropped for the drainage dummies in each crop because a class was absent in a small year; the per-year table's "slope_per_10mm" label is the module's column name, the unit here is per +1 pp.)

**Part 3, exploratory: both crops qualified (corn 5 drought years 2020–2024, soybeans 6 including 2018) and every range includes zero.** The 2017 no-till share and the 2017 cover-crop share show no consistent pattern: both lean slightly positive for corn and slightly negative for soybeans, with placebo estimates at zero. As written in advance, this cannot show a practice effect in either direction: counties may have adopted practices because of earlier droughts such as 2012 (Bueno de Mesquita & Fowler, ch. 9), and farmers choose these practices, so adopters differ from non-adopters in ways the controls cannot capture. The controls were restricted to longitude, latitude and slope; organic matter and storage were deliberately not controlled for because they lie on the pathway from practice to drought loss (Bueno de Mesquita & Fowler, ch. 10; Gelman, Hill & Vehtari, §19.6). Five or six test years is also very few.

**On the ranges that exclude zero.** The two exploratory ranges rest on 13 and 14 years and were selected from 18 tests. Estimates that clear a threshold on few years tend to overstate the true size (Gelman, Hill & Vehtari, §4.4, §16.1); their magnitudes should not be quoted as effects. The same applies to the width of the primary ranges, which is driven by single years (for example the soybean 2018 drought slope of −15.8 on 17 counties and the corn 2013 slope of −8.7).

**Comparison with notebook 05's storage result.** Storage leaned toward smaller penalties for both crops (corn +0.9 pp per +10 mm with longitude, soybeans +1.4) with every range including zero and a clean placebo. Organic matter repeats that picture for corn and shows nothing for soybeans. Taken together, no soil property tested here, nor storage, is separated from zero in drought years, while every placebo sits at zero. Cross-county comparisons of this kind, on 13–14 drought years, do not have the resolution to confirm a soil buffer of the size these estimates suggest, and they cannot speak to what a practice would do on a given field. That leaves the ranking on the drought penalty itself, and leaves program effects to be measured directly, as in the next steps of the planner.

**Guardrails kept.** No specification, unit, threshold or control set changed after the results. `soil_planner/soil_tool.py`, the app, the ranking and the evidence tables are unchanged; the extra soil and practice data remain display-only elsewhere. Nothing under `data/` or `src/` was modified.